# Stock Price Predictor
### GRU + Multi-Head Attention Neural Network

A deep learning model that predicts stock prices with uncertainty quantification using quantile regression.

---
## 1. Setup

In [ ]:
!pip install -q tensorflow yfinance pandas numpy scikit-learn matplotlib plotly

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

---
## 2. Configuration

In [ ]:
CONFIG = {
    'SEQUENCE_LENGTH': 100,
    'HORIZON': 30,
    'TRAIN_DAYS': 730,
    'MAX_TICKERS': 50,
    'MIN_PRICE': 5,
    'BATCH_SIZE': 64,
    'EPOCHS': 20,
    'SEED': 42,
    'QUANTILES': [0.1, 0.5, 0.9]
}

np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

In [ ]:
SP500_TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
    'V', 'XOM', 'JPM', 'PG', 'MA', 'HD', 'CVX', 'MRK', 'ABBV', 'LLY',
    'PEP', 'KO', 'COST', 'AVGO', 'WMT', 'MCD', 'CSCO', 'TMO', 'ACN', 'ABT',
    'DHR', 'NEE', 'LIN', 'ADBE', 'NKE', 'CRM', 'TXN', 'PM', 'WFC', 'BMY',
    'UPS', 'RTX', 'ORCL', 'MS', 'QCOM', 'T', 'UNP', 'INTC', 'HON', 'LOW'
]

---
## 3. Data Collection

In [ ]:
def fetch_stock_data(ticker, days=730):
    try:
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)
        stock = yf.Ticker(ticker)
        df = stock.history(start=start_date, end=end_date)
        if df.empty or len(df) < CONFIG['SEQUENCE_LENGTH'] + CONFIG['HORIZON'] + 10:
            return None
        return df['Close'].values.astype(np.float32)
    except:
        return None

In [ ]:
def collect_training_data(tickers, days=730):
    all_series = []
    successful = []
    
    for i, ticker in enumerate(tickers):
        series = fetch_stock_data(ticker, days)
        if series is not None and np.nanmin(series) >= CONFIG['MIN_PRICE']:
            all_series.append((ticker, series))
            successful.append(ticker)
        if (i + 1) % 10 == 0:
            print(f"Progress: {i+1}/{len(tickers)} | Collected: {len(successful)}")
    
    print(f"\nTotal collected: {len(successful)} tickers")
    return all_series, successful

all_series, successful_tickers = collect_training_data(SP500_TICKERS[:CONFIG['MAX_TICKERS']], CONFIG['TRAIN_DAYS'])

---
## 4. Data Visualization

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=('Sample Stock Prices', 'Price Distributions', 'Data Points per Stock', 'Price Ranges'))

colors = px.colors.qualitative.Set2
for i, (ticker, series) in enumerate(all_series[:8]):
    fig.add_trace(go.Scatter(y=series, name=ticker, line=dict(color=colors[i % len(colors)], width=1.5)), row=1, col=1)

all_prices = np.concatenate([s for _, s in all_series])
fig.add_trace(go.Histogram(x=all_prices, nbinsx=50, name='Price Distribution', marker_color='#6366f1'), row=1, col=2)

data_lengths = [len(s) for _, s in all_series]
fig.add_trace(go.Bar(x=successful_tickers, y=data_lengths, name='Data Points', marker_color='#10b981'), row=2, col=1)

price_ranges = [(ticker, np.min(s), np.max(s)) for ticker, s in all_series]
for ticker, min_p, max_p in price_ranges[:15]:
    fig.add_trace(go.Scatter(x=[ticker, ticker], y=[min_p, max_p], mode='lines+markers', line=dict(width=3), marker=dict(size=8), showlegend=False), row=2, col=2)

fig.update_layout(height=700, title_text="Training Data Overview", showlegend=True)
fig.update_xaxes(title_text="Days", row=1, col=1)
fig.update_xaxes(title_text="Price ($)", row=1, col=2)
fig.update_xaxes(title_text="Ticker", row=2, col=1, tickangle=45)
fig.update_xaxes(title_text="Ticker", row=2, col=2, tickangle=45)
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Days", row=2, col=1)
fig.update_yaxes(title_text="Price ($)", row=2, col=2)
fig.show()

In [ ]:
returns_data = []
for ticker, series in all_series:
    returns = np.diff(series) / series[:-1] * 100
    returns_data.append({'Ticker': ticker, 'Mean Return': np.mean(returns), 'Volatility': np.std(returns), 'Current Price': series[-1]})

returns_df = pd.DataFrame(returns_data)

fig = px.scatter(returns_df, x='Volatility', y='Mean Return', size='Current Price', color='Current Price',
                 hover_name='Ticker', color_continuous_scale='Viridis', title='Risk vs Return Analysis')
fig.update_layout(height=500)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.show()

---
## 5. Data Preprocessing

In [ ]:
def build_sequences(series, sequence_length, horizon):
    X, y = [], []
    for i in range(len(series) - sequence_length - horizon + 1):
        X.append(series[i:i + sequence_length])
        y.append(series[i + sequence_length:i + sequence_length + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def prepare_training_data(all_series, sequence_length, horizon):
    X_all, y_all = [], []
    scalers = {}
    ticker_labels = []
    
    for ticker, series in all_series:
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten().astype(np.float32)
        X, y = build_sequences(scaled, sequence_length, horizon)
        if len(X) > 0:
            X_all.append(X)
            y_all.append(y)
            scalers[ticker] = scaler
            ticker_labels.extend([ticker] * len(X))
    
    X = np.concatenate(X_all).astype(np.float32)
    y = np.concatenate(y_all).astype(np.float32)
    indices = np.random.permutation(len(X))
    return X[indices], y[indices], scalers, np.array(ticker_labels)[indices]

In [ ]:
X, y, scalers, ticker_labels = prepare_training_data(all_series, CONFIG['SEQUENCE_LENGTH'], CONFIG['HORIZON'])

split_idx = int(len(X) * 0.85)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]
ticker_val = ticker_labels[split_idx:]

print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Input shape: {X_train.shape}")
print(f"Output shape: {y_train.shape}")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Sample Input Sequence', 'Corresponding Target'))

sample_idx = 0
fig.add_trace(go.Scatter(y=X_train[sample_idx].flatten(), mode='lines', name='Input (100 days)', line=dict(color='#2563eb')), row=1, col=1)
fig.add_trace(go.Scatter(y=y_train[sample_idx], mode='lines+markers', name='Target (30 days)', line=dict(color='#10b981')), row=1, col=2)

fig.update_layout(height=350, title_text="Sample Training Data (Normalized)")
fig.update_xaxes(title_text="Days", row=1, col=1)
fig.update_xaxes(title_text="Days", row=1, col=2)
fig.update_yaxes(title_text="Normalized Price", row=1, col=1)
fig.update_yaxes(title_text="Normalized Price", row=1, col=2)
fig.show()

---
## 6. Model Architecture

In [ ]:
QUANTILES = CONFIG['QUANTILES']

def quantile_loss(y_true, y_pred):
    losses = []
    for q_index, q in enumerate(QUANTILES):
        errors = y_true - y_pred[:, :, q_index]
        losses.append(tf.maximum(q * errors, (q - 1) * errors))
    return tf.reduce_mean(tf.add_n(losses))

def median_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred[:, :, 1]))

def median_mape(y_true, y_pred):
    return tf.reduce_mean(tf.abs((y_true - y_pred[:, :, 1]) / tf.maximum(y_true, 1e-6))) * 100

In [ ]:
def build_model(sequence_length, horizon):
    inputs = keras.layers.Input(shape=(sequence_length, 1))
    x = keras.layers.GRU(64, return_sequences=True)(inputs)
    x = keras.layers.Dropout(0.25)(x)
    attn = keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = keras.layers.Add()([x, attn])
    x = keras.layers.LayerNormalization()(x)
    x = keras.layers.GRU(64)(x)
    x = keras.layers.Dropout(0.25)(x)
    outputs = keras.layers.Dense(horizon * len(QUANTILES))(x)
    outputs = keras.layers.Reshape((horizon, len(QUANTILES)))(outputs)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=quantile_loss, metrics=[median_mae, median_mape])
    return model

model = build_model(CONFIG['SEQUENCE_LENGTH'], CONFIG['HORIZON'])
model.summary()

---
## 7. Training

In [ ]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=CONFIG['EPOCHS'], batch_size=CONFIG['BATCH_SIZE'], callbacks=callbacks, verbose=1)

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=('Loss', 'MAE', 'MAPE'))

fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss', line=dict(color='#2563eb')), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Val Loss', line=dict(color='#f59e0b')), row=1, col=1)

fig.add_trace(go.Scatter(y=history.history['median_mae'], name='Train MAE', line=dict(color='#2563eb')), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_median_mae'], name='Val MAE', line=dict(color='#f59e0b')), row=1, col=2)

fig.add_trace(go.Scatter(y=history.history['median_mape'], name='Train MAPE', line=dict(color='#2563eb')), row=1, col=3)
fig.add_trace(go.Scatter(y=history.history['val_median_mape'], name='Val MAPE', line=dict(color='#f59e0b')), row=1, col=3)

fig.update_layout(height=350, title_text="Training History")
fig.update_xaxes(title_text="Epoch")
fig.show()

print(f"\nFinal Val Loss: {history.history['val_loss'][-1]:.4f}")
print(f"Final Val MAE: {history.history['val_median_mae'][-1]:.4f}")

---
## 8. Model Evaluation

In [ ]:
val_preds = model.predict(X_val, verbose=0)

fig = make_subplots(rows=2, cols=2, subplot_titles=('Predicted vs Actual (Day 1)', 'Predicted vs Actual (Day 15)', 'Predicted vs Actual (Day 30)', 'Prediction Error Distribution'))

for idx, day in enumerate([0, 14, 29]):
    row, col = (idx // 2) + 1, (idx % 2) + 1
    actual = y_val[:, day]
    predicted = val_preds[:, day, 1]
    fig.add_trace(go.Scatter(x=actual, y=predicted, mode='markers', marker=dict(size=3, opacity=0.5), name=f'Day {day+1}'), row=row, col=col)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash', color='red'), showlegend=False), row=row, col=col)

errors = (val_preds[:, :, 1] - y_val).flatten()
fig.add_trace(go.Histogram(x=errors, nbinsx=50, marker_color='#6366f1', name='Errors'), row=2, col=2)

fig.update_layout(height=600, title_text="Model Evaluation")
fig.show()

In [ ]:
day_errors = []
for day in range(CONFIG['HORIZON']):
    mae = np.mean(np.abs(y_val[:, day] - val_preds[:, day, 1]))
    day_errors.append(mae)

fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(1, 31)), y=day_errors, marker_color='#6366f1'))
fig.update_layout(title='Prediction Error by Forecast Day', xaxis_title='Days Ahead', yaxis_title='MAE (Normalized)', height=400)
fig.show()

---
## 9. Make Predictions

In [ ]:
def predict_stock(ticker, days_ahead=30):
    days_ahead = min(days_ahead, CONFIG['HORIZON'])
    series = fetch_stock_data(ticker, days=365)
    if series is None:
        raise ValueError(f"Could not fetch data for {ticker}")
    if len(series) < CONFIG['SEQUENCE_LENGTH']:
        raise ValueError(f"Insufficient data for {ticker}")
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten().astype(np.float32)
    last_sequence = scaled[-CONFIG['SEQUENCE_LENGTH']:]
    input_data = np.zeros((1, CONFIG['SEQUENCE_LENGTH'], 1), dtype=np.float32)
    input_data[0, :, 0] = last_sequence
    
    input_tensor = tf.constant(input_data, dtype=tf.float32)
    pred = model(input_tensor, training=False).numpy()
    
    pred_lower = scaler.inverse_transform(pred[0, :days_ahead, 0].reshape(-1, 1)).flatten()
    pred_median = scaler.inverse_transform(pred[0, :days_ahead, 1].reshape(-1, 1)).flatten()
    pred_upper = scaler.inverse_transform(pred[0, :days_ahead, 2].reshape(-1, 1)).flatten()
    
    current_price = float(series[-1])
    predicted_price = float(pred_median[0])
    
    return {
        'ticker': ticker,
        'current_price': current_price,
        'predicted_price': predicted_price,
        'price_change': predicted_price - current_price,
        'price_change_percent': ((predicted_price - current_price) / current_price) * 100,
        'predictions': {'lower': pred_lower.tolist(), 'median': pred_median.tolist(), 'upper': pred_upper.tolist()},
        'recent_prices': series[-60:].tolist(),
        'days_ahead': days_ahead
    }

In [ ]:
def visualize_prediction(result):
    ticker = result['ticker']
    recent = result['recent_prices']
    pred_median = result['predictions']['median']
    pred_lower = result['predictions']['lower']
    pred_upper = result['predictions']['upper']
    
    historical_days = list(range(-len(recent) + 1, 1))
    future_days = list(range(1, len(pred_median) + 1))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=historical_days, y=recent, mode='lines', name='Historical', line=dict(color='#2563eb', width=2)))
    fig.add_trace(go.Scatter(x=future_days + future_days[::-1], y=pred_upper + pred_lower[::-1], fill='toself', fillcolor='rgba(99, 102, 241, 0.2)', line=dict(color='rgba(255,255,255,0)'), name='80% Confidence', hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=future_days, y=pred_median, mode='lines+markers', name='Predicted', line=dict(color='#10b981', width=2, dash='dash'), marker=dict(size=6)))
    fig.add_trace(go.Scatter(x=[0], y=[result['current_price']], mode='markers', name='Current', marker=dict(color='#f59e0b', size=12, symbol='star')))
    
    change_symbol = '▲' if result['price_change'] >= 0 else '▼'
    fig.update_layout(
        title=f"{ticker} Price Prediction<br><sup>${result['current_price']:.2f} → ${result['predicted_price']:.2f} ({change_symbol}{abs(result['price_change_percent']):.1f}%)</sup>",
        xaxis_title='Days (0 = Today)', yaxis_title='Price ($)', hovermode='x unified', template='plotly_white', height=500
    )
    fig.add_vline(x=0, line_dash='dot', line_color='gray', opacity=0.5)
    fig.show()

In [ ]:
TICKER = 'AAPL'
result = predict_stock(TICKER, days_ahead=30)
visualize_prediction(result)

In [ ]:
tickers_to_predict = ['MSFT', 'GOOGL', 'TSLA', 'NVDA']
all_results = []

for ticker in tickers_to_predict:
    try:
        result = predict_stock(ticker, days_ahead=30)
        all_results.append(result)
        visualize_prediction(result)
    except Exception as e:
        print(f"Error: {ticker} - {e}")

---
## 10. Multi-Stock Comparison

In [ ]:
comparison_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META']
comparison_results = []

for ticker in comparison_tickers:
    try:
        result = predict_stock(ticker, days_ahead=30)
        comparison_results.append(result)
    except:
        pass

fig = make_subplots(rows=2, cols=3, subplot_titles=[r['ticker'] for r in comparison_results])

for i, result in enumerate(comparison_results):
    row, col = (i // 3) + 1, (i % 3) + 1
    recent = result['recent_prices'][-30:]
    pred = result['predictions']['median']
    
    fig.add_trace(go.Scatter(y=recent, mode='lines', name='Historical', line=dict(color='#2563eb'), showlegend=(i==0)), row=row, col=col)
    fig.add_trace(go.Scatter(x=list(range(len(recent), len(recent) + len(pred))), y=pred, mode='lines', name='Predicted', line=dict(color='#10b981', dash='dash'), showlegend=(i==0)), row=row, col=col)

fig.update_layout(height=500, title_text="Multi-Stock Predictions")
fig.show()

In [ ]:
summary_data = []
for r in comparison_results:
    summary_data.append({
        'Ticker': r['ticker'],
        'Current ($)': f"{r['current_price']:.2f}",
        'Predicted ($)': f"{r['predicted_price']:.2f}",
        'Change (%)': f"{r['price_change_percent']:+.1f}%",
        'Direction': '📈' if r['price_change'] > 0 else '📉'
    })

summary_df = pd.DataFrame(summary_data)
summary_df

In [ ]:
changes = [r['price_change_percent'] for r in comparison_results]
tickers = [r['ticker'] for r in comparison_results]
colors = ['#10b981' if c > 0 else '#ef4444' for c in changes]

fig = go.Figure(go.Bar(x=tickers, y=changes, marker_color=colors, text=[f"{c:+.1f}%" for c in changes], textposition='outside'))
fig.update_layout(title='Predicted Price Change (%)', xaxis_title='Ticker', yaxis_title='Change (%)', height=400)
fig.add_hline(y=0, line_color='gray', line_dash='dash')
fig.show()

---
## 11. Save Model

In [ ]:
model.save('stock_predictor_model.keras')
print("Model saved!")

try:
    from google.colab import files
    files.download('stock_predictor_model.keras')
except:
    pass

---

**Disclaimer:** Predictions are for educational purposes only. Not financial advice.